# Sparkov M4 — CatBoost + Isolation Forest (Feature-Level Integration)

**Selected baseline:** M1 static cost-sensitive CatBoost (validation PR-AUC **0.989228**). M3 pseudo-labelled weekly CatBoost (0.988218) did not beat M1, although it improved on its matched verified-only control (0.987465).

**M4 hypothesis:** An unsupervised Isolation Forest anomaly score may provide incremental fraud-ranking information beyond the original CatBoost features. This notebook implements **feature-level integration**, consistent with the report methodology, and separately measures the standalone predictive value of `iforest_score`.

**Runtime:** One Isolation Forest fit and one M4 CatBoost fit. The saved M1 predictions are reused. **Do not rerun Notebook 06.**

## 1. Load processed Sparkov data and selected M1 validation predictions

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.ensemble import IsolationForest
from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score, roc_auc_score

DATA_DIR = Path('../data/processed')
M3_RESULTS_DIR = Path('../results/m3_catboost_selected_m1')
RESULTS_DIR = Path('../results/m4_catboost_isolation_forest')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

TIME_COL = 'trans_date_trans_time'
TARGET_COL = 'is_fraud'
SPLIT_COL = 'sparkov_split'
IFOREST_SCORE_COL = 'iforest_score'

frame = pd.read_pickle(DATA_DIR / 'sparkov_features.pkl')
with open(DATA_DIR / 'feature_lists.json', 'r') as f:
    feature_cols = json.load(f)['sparkov_feature_cols']
frame[TIME_COL] = pd.to_datetime(frame[TIME_COL])
# Do not sort: preserve the same M1 row ordering as Notebook 05/06.
frame = frame.loc[frame[SPLIT_COL].isin(['train', 'validation'])].copy()
train_rows = frame.loc[frame[SPLIT_COL].eq('train')].copy()
val_rows = frame.loc[frame[SPLIT_COL].eq('validation')].copy()
assert train_rows[TIME_COL].max() < val_rows[TIME_COL].min(), 'Chronological split overlaps'

categorical_cols = train_rows[feature_cols].select_dtypes(include=['object','category','string']).columns.tolist()
assert IFOREST_SCORE_COL not in feature_cols

m1_path = M3_RESULTS_DIR / 'm1_catboost_validation_predictions.csv'
if not m1_path.exists():
    raise FileNotFoundError(f'Missing saved M1 predictions: {m1_path}. Run Notebook 06 first; do not refit M1 here.')
m1_pred = pd.read_csv(m1_path)
assert {'row_index','y_true','y_score'}.issubset(m1_pred.columns)
assert len(m1_pred) == len(val_rows)
assert m1_pred['row_index'].is_unique
assert set(m1_pred['row_index'].astype(int)) == set(val_rows.index)
assert np.array_equal(m1_pred.set_index('row_index').loc[val_rows.index, 'y_true'].astype(int).to_numpy(), val_rows[TARGET_COL].astype(int).to_numpy())
m1_ap = average_precision_score(m1_pred['y_true'], m1_pred['y_score'])
assert np.isclose(m1_ap, 0.989228, atol=0.0001), f'Unexpected M1 PR-AUC: {m1_ap:.6f}'
print(f'Train: {len(train_rows):,}; validation: {len(val_rows):,}; features: {len(feature_cols)}')
print(f'Saved M1 validation PR-AUC: {m1_ap:.6f}')

Train: 1,037,340; validation: 259,335; features: 62
Saved M1 validation PR-AUC: 0.989228


## 2. Fit Isolation Forest on training features only

Use the numeric subset of the **approved modelling features**. Fit median imputation and Isolation Forest on at most 100,000 training transactions, without labels. `iforest_score = -score_samples(...)` means **higher = more anomalous**. Score both training and validation transactions using the same fitted detector. No validation labels or test transactions are used for fitting.

In [2]:
IFOREST_SETTINGS = dict(n_estimators=200, max_samples=256, random_state=100, n_jobs=2)
IFOREST_FIT_CAP = 100_000
anomaly_cols = [c for c in feature_cols if pd.api.types.is_numeric_dtype(train_rows[c])]
assert anomaly_cols, 'No numeric predictors for Isolation Forest'

iforest_fit_rows = train_rows.sample(n=min(IFOREST_FIT_CAP, len(train_rows)), random_state=100)
fit_numeric = iforest_fit_rows[anomaly_cols].replace([np.inf, -np.inf], np.nan)
usable_cols = fit_numeric.columns[fit_numeric.notna().any(axis=0)].tolist()
assert usable_cols
imputer = SimpleImputer(strategy='median')
X_if_fit = imputer.fit_transform(fit_numeric[usable_cols]).astype('float32')
forest = IsolationForest(**IFOREST_SETTINGS)
forest.fit(X_if_fit)

def anomaly_scores(rows):
    x = rows[usable_cols].replace([np.inf, -np.inf], np.nan)
    return -forest.score_samples(imputer.transform(x).astype('float32'))

train_rows[IFOREST_SCORE_COL] = anomaly_scores(train_rows)
val_rows[IFOREST_SCORE_COL] = anomaly_scores(val_rows)
assert np.isfinite(train_rows[IFOREST_SCORE_COL]).all()
assert np.isfinite(val_rows[IFOREST_SCORE_COL]).all()
print(f'Isolation Forest fitted on {len(iforest_fit_rows):,} training rows, {len(usable_cols)} numeric features.')
print('iforest_score: higher means more anomalous')

Isolation Forest fitted on 100,000 training rows, 59 numeric features.
iforest_score: higher means more anomalous


## 3. Standalone Isolation Forest evaluation

Evaluate the anomaly score **without CatBoost**. This is a diagnostic, not a replacement for M1. Report PR-AUC, ROC-AUC, and observed fraud rates across anomaly-score deciles. Decile 10 contains the most anomalous transactions.

In [3]:
y_val = val_rows[TARGET_COL].astype(int).to_numpy()
iforest_val_score = val_rows[IFOREST_SCORE_COL].to_numpy()
iforest_metrics = pd.DataFrame([{
    'model': 'Isolation Forest standalone',
    'validation_rows': len(val_rows),
    'fraud_cases': int(y_val.sum()),
    'pr_auc': average_precision_score(y_val, iforest_val_score),
    'roc_auc': roc_auc_score(y_val, iforest_val_score),
    'fraud_rate_baseline': float(y_val.mean()),
}])
display(iforest_metrics)

iforest_deciles = pd.DataFrame({'iforest_score': iforest_val_score, 'is_fraud': y_val})
# Rank ensures ten approximately equal-sized groups even if scores tie.
iforest_deciles['anomaly_decile'] = pd.qcut(iforest_deciles['iforest_score'].rank(method='first'), 10, labels=False).astype(int) + 1
iforest_decile_summary = iforest_deciles.groupby('anomaly_decile', as_index=False).agg(
    transactions=('is_fraud','size'), fraud_cases=('is_fraud','sum'),
    fraud_rate=('is_fraud','mean'), mean_iforest_score=('iforest_score','mean'))
display(iforest_decile_summary)
iforest_metrics.to_csv(RESULTS_DIR / 'iforest_standalone_metrics.csv', index=False)
iforest_decile_summary.to_csv(RESULTS_DIR / 'iforest_fraud_by_decile.csv', index=False)

,model,validation_rows,fraud_cases,pr_auc,roc_auc,fraud_rate_baseline
0,Isolation Forest standalone,259335,1538,0.612183,0.970196,0.005931


,anomaly_decile,transactions,fraud_cases,fraud_rate,mean_iforest_score
0,1,25934,2,0.000077,0.408683
1,2,25933,1,0.000039,0.418961
2,3,25934,2,0.000077,0.425857
3,4,25933,5,0.000193,0.432269
4,5,25934,6,0.000231,0.438724
5,6,25933,13,0.000501,0.445611
6,7,25933,15,0.000578,0.453286
7,8,25934,28,0.001080,0.462601
8,9,25933,53,0.002044,0.475802
9,10,25934,1413,0.054484,0.511974


## 4. Train M4 CatBoost with the additional anomaly-score feature

**Only difference from M1:** append `iforest_score` to the original features. Match M1 CatBoost hyperparameters and categorical preprocessing. Fit on the original training split, predict once on the untouched chronological validation split. This is a **static** M4 comparison because the selected baseline M1 is static.

In [4]:
m4_feature_cols = feature_cols + [IFOREST_SCORE_COL]

def prepare_catboost(rows, columns):
    X = rows[columns].copy()
    for col in categorical_cols:
        X[col] = X[col].fillna('Missing').astype(str)
    return X

m4_model = CatBoostClassifier(
    iterations=500, learning_rate=0.1, depth=6,
    loss_function='Logloss', cat_features=categorical_cols,
    auto_class_weights='Balanced', thread_count=4,
    random_seed=100, verbose=False, allow_writing_files=False,
)
m4_model.fit(prepare_catboost(train_rows, m4_feature_cols), train_rows[TARGET_COL].astype(int))
m4_scores = m4_model.predict_proba(prepare_catboost(val_rows, m4_feature_cols))[:,1]
m4_pred = pd.DataFrame({
    'row_index': val_rows.index.to_numpy(),
    'timestamp': val_rows[TIME_COL].to_numpy(),
    'y_true': y_val,
    'y_score': m4_scores,
    IFOREST_SCORE_COL: iforest_val_score,
})
m4_pred.to_csv(RESULTS_DIR / 'm4_validation_predictions.csv', index=False)
print('M4 validation PR-AUC:', f'{average_precision_score(y_val, m4_scores):.6f}')

M4 validation PR-AUC: 0.986560


## 5. M1 vs M4 validation selection and next-stage handoff

In [5]:
m1_ordered = m1_pred.set_index('row_index').loc[val_rows.index]
m1_scores = m1_ordered['y_score'].to_numpy()
assert np.array_equal(m1_ordered['y_true'].astype(int).to_numpy(), y_val)
comparison = pd.DataFrame([
    {'stage':'M1', 'model':'Static CatBoost', 'strategy':'Original approved features',
     'pr_auc':average_precision_score(y_val,m1_scores), 'roc_auc':roc_auc_score(y_val,m1_scores)},
    {'stage':'M4', 'model':'Static CatBoost + Isolation Forest', 'strategy':'Original features + iforest_score',
     'pr_auc':average_precision_score(y_val,m4_scores), 'roc_auc':roc_auc_score(y_val,m4_scores)},
]).sort_values('pr_auc', ascending=False).reset_index(drop=True)
display(comparison)
winner = comparison.iloc[0]
selected_stage = str(winner['stage'])
print(f'SELECTED FOR NOTEBOOK 08: {selected_stage} | validation PR-AUC={winner["pr_auc"]:.6f}')
print(f'M4 minus M1 PR-AUC: {average_precision_score(y_val,m4_scores)-m1_ap:+.6f}')
comparison.to_csv(RESULTS_DIR / 'm4_validation_comparison.csv', index=False)
pd.DataFrame([{
    'selected_stage': selected_stage, 'model': str(winner['model']),
    'selected_validation_pr_auc': float(winner['pr_auc']),
    'm1_validation_pr_auc': float(m1_ap),
    'm4_validation_pr_auc': float(average_precision_score(y_val,m4_scores)),
    'iforest_standalone_pr_auc': float(iforest_metrics.loc[0,'pr_auc']),
    'iforest_score_col': IFOREST_SCORE_COL,
    'iforest_n_estimators': IFOREST_SETTINGS['n_estimators'],
    'iforest_fit_cap': IFOREST_FIT_CAP,
    'iforest_seed': IFOREST_SETTINGS['random_state'],
}]).to_csv(RESULTS_DIR / 'selected_stage_after_m4.csv', index=False)

,stage,model,strategy,pr_auc,roc_auc
0,M1,Static CatBoost,Original approved features,0.989228,0.999875
1,M4,Static CatBoost + Isolation Forest,Original features + iforest_score,0.986560,0.999764


SELECTED FOR NOTEBOOK 08: M1 | validation PR-AUC=0.989228
M4 minus M1 PR-AUC: -0.002668



## 6. Interpretation and Final-Test Handoff

### 6.1 Isolation Forest Evaluation

Standalone Isolation Forest achieved a validation PR-AUC of 0.612183 and ROC-AUC of 0.970196. The highest anomaly-score decile contained 1,413 of the 1,538 validation fraud cases (91.9%), indicating that anomaly scores capture meaningful fraud-related patterns.

However, standalone Isolation Forest performed substantially below the supervised CatBoost baseline.

### 6.2 M4 Hybrid Model Performance

Adding `iforest_score` as an additional CatBoost feature produced a validation PR-AUC of 0.986560, compared with 0.989228 for M1 static CatBoost.

The decrease of 0.002668 indicates that feature-level anomaly-score integration did not improve the supervised model's fraud ranking.

### 6.3 Final Model Selection

M1 static CatBoost achieved the highest validation PR-AUC (0.989228), outperforming M3 pseudo-labelled CatBoost (0.988218) and M4 CatBoost with Isolation Forest (0.986560).

Therefore, M1 static CatBoost is selected for final held-out test evaluation.

### 6.4 Final-Test Handoff

Notebook 08 will evaluate the selected M1 CatBoost configuration on the held-out test set.

The evaluation will report PR-AUC, ROC-AUC, precision and recall at threshold 0.5, Precision@100 per day, and operational alert and cost metrics.

The daily alert capacity K will be selected using validation data only and then applied unchanged to the test set. Test results will not be used to reconsider model selection.
